<a href="https://colab.research.google.com/github/eusmani/ML-basics/blob/main/house_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# Data source: Ames Housing Dataset
# https://github.com/MatthewChatham/ames/blob/master/train.csv

url = "https://raw.githubusercontent.com/MatthewChatham/ames/master/train.csv"
df = pd.read_csv(url)
# Select relevant columns
df = df[["GrLivArea", "Neighborhood", "SalePrice"]]

# Rename columns to match the assignment
df = df.rename(columns={
    "GrLivArea": "square_footage",
    "Neighborhood": "location",
    "SalePrice": "price"
})

# Remove missing values
df = df.dropna()

print("Number of records:", len(df))
print(df.head())

# Features and target
X = df[["square_footage", "location"]]
y = df["price"]

# One-hot encode the location column
preprocessor = ColumnTransformer(
    transformers=[
        (
            "location",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ["location"]
        )
    ],
    remainder="passthrough"
)

# Create the pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Test the model
predictions = model.predict(X_test)

print("\nModel Performance:")
print("Mean Absolute Error: ${:,.2f}".format(
    mean_absolute_error(y_test, predictions)
))
print("R-squared Score: {:.3f}".format(
    r2_score(y_test, predictions)
))

# Predict a 2,000-square-foot house
# NridgHt is an actual neighborhood in the dataset
new_house = pd.DataFrame({
    "square_footage": [2000],
    "location": ["NridgHt"]
})

predicted_price = model.predict(new_house)[0]

print("\nPredicted price for a 2,000-square-foot house in NridgHt:")
print("${:,.2f}".format(predicted_price))

# Display coefficients
feature_names = model.named_steps[
    "preprocessor"
].get_feature_names_out()

coefficients = model.named_steps["regressor"].coef_

print("\nModel Coefficients:")
for feature, coefficient in zip(feature_names, coefficients):
    print(f"{feature}: ${coefficient:,.2f}")

Number of records: 1460
   square_footage location   price
0            1710  CollgCr  208500
1            1262  Veenker  181500
2            1786  CollgCr  223500
3            1717  Crawfor  140000
4            2198  NoRidge  250000

Model Performance:
Mean Absolute Error: $28,298.36
R-squared Score: 0.757

Predicted price for a 2,000-square-foot house in NridgHt:
$317,342.31

Model Coefficients:
location__location_Blmngtn: $15,916.57
location__location_Blueste: $-36,576.67
location__location_BrDale: $-51,780.44
location__location_BrkSide: $-34,350.03
location__location_ClearCr: $16,301.31
location__location_CollgCr: $19,492.68
location__location_Crawfor: $7,062.28
location__location_Edwards: $-40,751.05
location__location_Gilbert: $2,269.27
location__location_IDOTRR: $-49,248.61
location__location_MeadowV: $-52,983.05
location__location_Mitchel: $-9,646.37
location__location_NAmes: $-20,084.98
location__location_NPkVill: $-17,811.23
location__location_NWAmes: $-9,598.79
location__loc